# Fraud risk analysis
Offline assessment on supplied data. Costs and review outcomes are assumptions, not observed business results.

## 1. Objective and assumptions
Score fraud probability, simulate three decisions, and minimize hypothetical cost subject to review capacity. No event timestamps or customer keys are available.

## 2. Environment and reproducibility
Seeds 42-45 control splitting and models. Execute from the repository root; raw CSV belongs in `data/raw/`.

In [1]:
from pathlib import Path
import json, pandas as pd, subprocess, sys
print('Python:', sys.version.split()[0])

Python: 3.11.9


## 3. Dataset loading
Running this notebook rebuilds analysis outputs from source, without regenerating the open notebook. Existing outputs are not silently reused.

In [2]:
source = 'D:\\GR - Intern\\Fraud Risk Pipeline\\fraud-risk-intelligence-pipeline\\data\\raw\\transaction_fraud_risk_advanced_dataset.csv'
if not globals().get('_pipeline_build_complete', False):
    subprocess.run([sys.executable, 'scripts/run_analysis.py', '--skip-notebook', '--data', source], check=True)
data = pd.read_csv(source)
print('Shape:', data.shape, 'Fraud:', int(data.fraud.sum()))

Shape: (3000, 25) Fraud: 364


## 4. Data audit
Schema, missingness, duplicate identifiers, and partition counts are machine-readable in `reports/data_audit.json`.

In [3]:
audit = json.loads(Path('reports/data_audit.json').read_text())
print(pd.Series(audit['missing']).sort_values(ascending=False).head(12).to_string())
print(audit['partitions'])

authentication_level               289
distance_from_usual_location_km     75
merchant_risk_score                 60
login_to_transaction_minutes        60
transaction_amount                  53
ip_risk_score                       45
transaction_id                       0
day_of_week                          0
hour                                 0
failed_logins_24h                    0
account_age_days                     0
transactions_last_24h                0
{'train': {'rows': 1500, 'fraud': 182}, 'selection': {'rows': 450, 'fraud': 55}, 'calibration_fit': {'rows': 225, 'fraud': 27}, 'policy_selection': {'rows': 225, 'fraud': 27}, 'test': {'rows': 600, 'fraud': 73}}


## 5. Leakage assessment
`fraud` and transaction IDs are excluded. Chargebacks, velocity, distance, and upstream scores require point-in-time lineage proof before deployment.

## 6. Exploratory analysis
Charts use training rows for fraud patterns. Full-data quality charts are descriptive. Interaction cells may have small sample sizes.

![Class distribution](reports/figure_01.png)

![Missingness](reports/figure_02.png)

![Channel fraud rates](reports/figure_04.png)

![Device-distance interaction](reports/figure_10.png)

![Merchant amount interaction](reports/figure_11.png)

## 7. Feature engineering
Eighteen derived features include training-fold merchant amount normalization and channel distance deviation. Population references are fitted inside each training pipeline; they are not personal customer histories.

In [4]:
print(pd.read_csv('processed_dataset.csv', nrows=1).columns.tolist())

['transaction_id', 'hour', 'day_of_week', 'transaction_amount', 'distance_from_usual_location_km', 'transactions_last_1h', 'transactions_last_24h', 'account_age_days', 'failed_logins_24h', 'prior_chargebacks', 'device_age_days', 'merchant_risk_score', 'ip_risk_score', 'email_age_days', 'login_to_transaction_minutes', 'card_presence', 'channel', 'country', 'device_type', 'merchant_category', 'authentication_level', 'new_device', 'billing_address_mismatch', 'customer_tier', 'log_amount', 'amount_risk', 'device_distance', 'distance_risk', 'velocity_ratio', 'velocity_amount', 'new_account', 'young_device', 'night_transaction', 'weekend', 'auth_channel', 'login_speed', 'chargeback_amount', 'velocity_burst', 'combined_risk', 'amount_to_merchant_median', 'amount_merchant_deviation', 'distance_channel_deviation', 'fraud']


## 8. Validation
Stratified train, model-selection, calibrator-fit, policy-selection, and final-test partitions are disjoint. They are not temporal validation.

In [5]:
print(pd.DataFrame(audit['partitions']).T.to_string())

                  rows  fraud
train             1500    182
selection          450     55
calibration_fit    225     27
policy_selection   225     27
test               600     73


## 9. Baseline modeling
Unweighted logistic regression with imputation, scaling, and one-hot encoding is the reference model.

In [6]:
comparison = pd.read_csv('reports/model_comparison.csv')
print(comparison[comparison.name.eq('logistic_plain_base')].to_string(index=False))

               name  roc_auc  average_precision    brier  log_loss
logistic_plain_base 0.660345           0.289115 0.101044  0.365075


## 10. Nonlinear models
Random forest and extra trees are compared under the same partition design.

In [7]:
print(comparison[comparison.name.str.startswith(('forest','extra_trees'))].head(6).to_string(index=False))

                           name  roc_auc  average_precision    brier  log_loss
   extra_trees_plain_engineered 0.651277           0.255542 0.103339  0.355307
        forest_tuned_engineered 0.651784           0.245559 0.103096  0.353589
              forest_plain_base 0.621082           0.228846 0.104434  0.359457
extra_trees_weighted_engineered 0.619102           0.228216 0.144378  0.464786
   extra_trees_tuned_engineered 0.633234           0.228058 0.104086  0.359801
        forest_plain_engineered 0.652704           0.220756 0.103296  0.353097


## 11. Hyperparameter tuning
Logistic C is searched in three stratified training folds. These folds do not establish temporal stability.

In [8]:
print(pd.read_csv('reports/stratified_cv_tuning.csv').to_string(index=False))

 param_model__C  mean_test_score  std_test_score  split0_test_score  split1_test_score  split2_test_score
           0.03         0.221365        0.028419           0.241709           0.241210           0.181175
           0.10         0.216515        0.031554           0.250165           0.225073           0.174307
           0.30         0.210888        0.035274           0.253172           0.212667           0.166824
           1.00         0.206865        0.034737           0.247766           0.209979           0.162850
           3.00         0.205409        0.035195           0.247588           0.207205           0.161434
          10.00         0.204078        0.033994           0.245297           0.204895           0.162041


## 12. Class imbalance
Unweighted, class-weighted, and training-only undersampled logistic models are compared on untouched selection rows.

In [9]:
print(pd.read_csv('reports/imbalance_comparison.csv').to_string(index=False))

                      name  roc_auc  average_precision    brier  log_loss
       logistic_plain_base 0.660345           0.289115 0.101044  0.365075
    logistic_weighted_base 0.645109           0.266017 0.229258  0.668007
logistic_undersampled_base 0.669183           0.232984 0.133839  0.451032


## 13. Calibration
Sigmoid and isotonic mappings fit on calibration-fit rows and are compared on policy-selection rows.

![Calibration](reports/figure_14.png)

In [10]:
print(pd.read_csv('reports/calibration_comparison.csv').to_string(index=False))
print(pd.read_csv('reports/calibration_risk_bands.csv').to_string(index=False))

  method  roc_auc  average_precision    brier  log_loss  calibration_in_large  calibration_slope  expected_calibration_error
 sigmoid 0.638982           0.220790 0.104184  0.360464             -0.001676           2.566945                    0.061409
isotonic 0.624673           0.185057 0.105291  0.627370             -0.002043           0.000665                    0.029770
  risk_band  count  fraud_rate  mean_probability
(0.1, 0.15]    212    0.113208          0.118270
(0.15, 0.2]     12    0.250000          0.173826
 (0.2, 0.3]      1    0.000000          0.217985


## 14. Cost and threshold selection
Joint threshold search uses hypothetical costs; missing amounts have an explicit fallback loss and cannot be automatically approved.

![Cost by review threshold](reports/figure_15.png)

In [11]:
thresholds = pd.read_csv('threshold_cost_analysis.csv')
valid = thresholds[thresholds.capacity_ok]
print(valid.loc[valid.groupby('cost_scenario').total_expected_cost.idxmin()][['cost_scenario','review_threshold','block_threshold','total_expected_cost']].to_string(index=False))

          cost_scenario  review_threshold  block_threshold  total_expected_cost
                   base          0.124193         0.132910             2872.356
  higher_block_friction          0.138239         1.000000             3342.188
higher_fraud_block_cost          0.124193         0.132910             2916.356
      higher_fraud_loss          0.000000         0.107485             4726.984
    higher_missing_loss          0.124193         0.132910             2872.356
     higher_review_cost          0.124193         0.132910             2971.356
 higher_review_friction          0.124193         0.132910             2956.356
   lower_review_success          0.124193         0.132910             3001.152
tighter_review_capacity          0.132910         0.138239             3072.786


## 15. Explainability
Permutation importance and local base-model log-odds contributions are associations, not calibrated-probability or causal contributions. Fold importance is stratified, not temporal.

In [12]:
print(pd.read_csv('reports/feature_importance.csv').head(10).to_string(index=False))
print(pd.read_csv('reports/feature_importance_folds.csv').groupby('feature').ap_drop.agg(['mean','std']).sort_values('mean',ascending=False).head(10).to_string())
print(pd.read_csv('reports/local_explanations.csv').head().to_string(index=False))

                 feature  ap_drop_mean  ap_drop_sd
       prior_chargebacks      0.057747    0.007382
              new_device      0.050912    0.006316
       merchant_category      0.049698    0.010903
    transactions_last_1h      0.042090    0.019408
                 channel      0.027903    0.015989
     merchant_risk_score      0.027493    0.015371
           customer_tier      0.026298    0.002043
billing_address_mismatch      0.020911    0.020498
    authentication_level      0.017837    0.008893
                 country      0.016687    0.009918
                                  mean       std
feature                                         
merchant_risk_score           0.034188  0.022720
new_device                    0.028308  0.030027
billing_address_mismatch      0.024610  0.014826
ip_risk_score                 0.018813  0.005630
channel                       0.012791  0.006002
prior_chargebacks             0.008081  0.007732
transaction_amount            0.003397  0.00182

## 16. Fraud operations
Review success and financial impacts are simulated; compare approve-all, 0.50 binary, review-only, and the selected policy.

In [13]:
results = json.loads(Path('reports/results.json').read_text())
print({k:round(results[k]['cost_per_transaction'],3) for k in ['approve_all','binary_050','review_only','test_policy']})

{'approve_all': 20.505, 'binary_050': 20.505, 'review_only': 14.352, 'test_policy': 15.36}


## 17. Robustness
Controlled perturbations retain labels and do not reproduce adaptive fraud.

In [14]:
print(pd.read_csv('reports/robustness.csv')[['scenario','average_precision','cost_per_transaction','review_count','failed']].to_string(index=False))

                  scenario  average_precision  cost_per_transaction  review_count  failed
                      base           0.211013             15.359877            79       0
         amount_plus_50pct           0.211503             13.800490            77       0
           velocity_plus_2           0.211013             14.774963           128       0
      missing_amount_10pct           0.199014             14.547673           126       0
            unseen_channel           0.196358             15.372713            94       0
     ecommerce_channel_mix           0.196358             15.835543           109       0
       extreme_top_amounts           0.223820             13.638793            80       0
fraud_prevalence_weight_2x           0.344523             26.631127            82       0


## 18. Final test
The final partition was evaluated after model, calibration, and policy selection.

![Precision-recall curve](reports/figure_13.png)

In [15]:
print(json.dumps(results['test_metrics'], indent=2))
print(json.dumps(results['test_policy'], indent=2))

{
  "roc_auc": 0.6514777364768267,
  "average_precision": 0.21101291795049917,
  "brier": 0.10527934587963679,
  "log_loss": 0.3632155847205391,
  "calibration_in_large": 0.0013672538443398469,
  "calibration_slope": 3.2972579347421753,
  "expected_calibration_error": 0.048763866746419304,
  "precision_at_review": 0.19883040935672514,
  "recall_at_review": 0.4657534246575342,
  "f1_at_review": 0.2786885245901639,
  "confusion_at_review": [
    [
      390,
      137
    ],
    [
      39,
      34
    ]
  ]
}
{
  "review_threshold": 0.12419255718132183,
  "block_threshold": 0.13290953546649942,
  "total_expected_cost": 9215.926,
  "cost_per_transaction": 15.359876666666667,
  "review_count": 79,
  "block_count": 92,
  "approve_count": 429,
  "legitimate_block_count": 71,
  "false_positive_count": 137,
  "fraud_approved_count": 39,
  "fraud_capture_rate": 0.4301369863013698,
  "precision": 0.19883040935672514,
  "recall": 0.4657534246575342,
  "review_capacity_utilization": 0.8777777777

## 19. Monitoring and governance
Monitor missingness, unseen categories, score and decision rates, review capacity, mature-label precision, calibration, and realized loss. Human approval and rollback procedures are required before live use.

## 20. Conclusions and limits
The dataset lacks event and label timestamps, customer keys, feature lineage, actual review outcomes, verified costs, and a fairness audit. No future-period or production-performance claim is supported.

### Added evidence and required business questions
# Evidence-based assessment



All model/cost comparisons below use development data. They are exploratory optimized costs, not unbiased savings estimates.



## Predictors and stability

             feature  ap_drop_mean  ap_drop_sd
   prior_chargebacks      0.057747    0.007382
          new_device      0.050912    0.006316
   merchant_category      0.049698    0.010903
transactions_last_1h      0.042090    0.019408
             channel      0.027903    0.015989



Fold estimates are in selected_baseline_importance_folds.csv. Negative permutation importance means the shuffled feature improved AP in that sample; correlated inputs and small samples can make rankings unstable. No causal claim follows.



Measured baseline stability (3 stratified folds):

                            mean     std     min     max  positive_folds
feature                                                                 
merchant_risk_score       0.0256  0.0166  0.0070  0.0386               3
new_device                0.0226  0.0223  0.0085  0.0483               3
billing_address_mismatch  0.0192  0.0219 -0.0009  0.0426               2
ip_risk_score             0.0131  0.0084  0.0078  0.0228               3
transactions_last_1h      0.0102  0.0097 -0.0006  0.0183               2
prior_chargebacks         0.0097  0.0086 -0.0001  0.0162               2
channel                   0.0094  0.0066  0.0021  0.0151               3
customer_tier             0.0043  0.0034  0.0010  0.0077               3

A feature positive in all three folds has more consistent direction in this experiment; overlapping uncertainty, only three folds, and unknown chronology prevent a claim of long-term stability.

## Model economics

          cost_scenario                      model  selection_ap  cost_per_transaction
                   base  logistic_plain_engineered      0.261445             12.520462
  higher_block_friction  logistic_plain_engineered      0.261445             14.631573
higher_fraud_block_cost  logistic_plain_engineered      0.261445             12.662684
      higher_fraud_loss forest_weighted_engineered      0.199165             16.833049
    higher_missing_loss  logistic_plain_engineered      0.261445             12.520462
     higher_review_cost  logistic_plain_engineered      0.261445             12.947129
 higher_review_friction  logistic_plain_engineered      0.261445             12.880462
   lower_review_success  logistic_plain_engineered      0.261445             13.091102
tighter_review_capacity  logistic_plain_engineered      0.261445             13.159849



Highest selection ROC-AUC candidate: logistic_plain_engineered. Compare its cost below with each scenario winner:

          cost_scenario  cost_per_transaction
                   base             12.520462
      higher_fraud_loss             19.883858
  higher_block_friction             14.631573
   lower_review_success             13.091102
     higher_review_cost             12.947129
 higher_review_friction             12.880462
higher_fraud_block_cost             12.662684
    higher_missing_loss             12.520462
tighter_review_capacity             13.159849

Measured examples where lower AP has lower optimized development cost:

         scenario             lower_ap_model  lower_ap  lower_ap_cost     higher_ap_model  higher_ap  higher_ap_cost  cost_advantage
higher_fraud_loss forest_weighted_engineered  0.199165      16.833049 logistic_tuned_base   0.252572       21.075271        4.242222
higher_fraud_loss forest_weighted_engineered  0.199165      16.833049 logistic_plain_base   0.289115       21.008818        4.175769
higher_fraud_loss       forest_weighted_base  0.183012      17.136267 logistic_tuned_base   0.252572       21.075271        3.939004
higher_fraud_loss    forest_plain_engineered  0.220756      17.143004 logistic_tuned_base   0.252572       21.075271        3.932267
higher_fraud_loss       forest_weighted_base  0.183012      17.136267 logistic_plain_base   0.289115       21.008818        3.872551



## Calibration

Calibration maps scores to empirical fraud frequencies on separate fit rows. A score of 0.10 should imply approximately 10 frauds per 100 comparable transactions only if validated calibration persists. Small calibration partitions and distribution shifts weaken this interpretation. See raw/sigmoid/isotonic Brier scores and calibration_methods.png.



## Threshold sensitivity

          cost_scenario  review_threshold  block_threshold  review_count  cost_per_transaction
                   base          0.124193         0.132910            33             12.766027
      higher_fraud_loss          0.000000         0.107485            26             21.008818
  higher_block_friction          0.138239         1.000000            31             14.854169
   lower_review_success          0.124193         0.132910            33             13.338453
     higher_review_cost          0.124193         0.132910            33             13.206027
 higher_review_friction          0.124193         0.132910            33             13.139360
higher_fraud_block_cost          0.124193         0.132910            33             12.961582
    higher_missing_loss          0.124193         0.132910            33             12.766027
tighter_review_capacity          0.132910         0.138239            18             13.656827



## Recall and review workload

Review-only threshold sweep (no model-driven blocking):

 review_threshold   recall  review_count  review_capacity_utilization
         0.000000 1.000000           225                     6.666667
         0.108360 0.777778           191                     5.659259
         0.112780 0.740741           146                     4.325926
         0.118811 0.629630            99                     2.933333
         0.129674 0.407407            52                     1.540741
         0.217985 0.000000             5                     0.148148

Higher recall requires more flagged transactions in this sweep; missing-amount review supplies a fixed workload floor. Capacity violations remain visible.



## Interactions

                                 interaction  eligible_cells  max_minus_min_rate
new_device x distance_from_usual_location_km               6            0.161287
    merchant_risk_score x transaction_amount               9            0.102182
              authentication_level x channel              11            0.192068

These are observed cell contrasts, not isolated interaction coefficients or causal effects. Counts and labeled heatmaps are supplied.



## Drift priorities

Prioritize the leading predictors above, transaction amount, velocity, authentication missingness, merchant/IP risk scores, channel mix, and output probabilities. Monitor point-in-time feature lineage and mature-label calibration. Upstream score changes, new merchants/devices, fraud adaptation, label delays, and changed review success can degrade performance.



## Counterintuitive explanations

Measured base-model dependence endpoints on development rows:

                                   first     last      min      max
feature                                                            
distance_from_usual_location_km  0.11882  0.13828  0.11882  0.13828
merchant_risk_score              0.09220  0.18055  0.09220  0.18055
transaction_amount               0.12397  0.12956  0.12397  0.12956
transactions_last_1h             0.08549  0.18472  0.08549  0.18472

A decreasing or flat amount/risk response can reflect weak signals, correlated fields, regularization, or sampling noise. These responses are model behavior, not evidence that increasing an amount or upstream risk score prevents fraud. Signed local terms are relative to encoded/scaled zero, not to an individual customer's normal behavior. A missing-input flag can be predictive of collection practices rather than fraud.



## Validation boundaries

The final model remains the average-precision-selected candidate; the economics table is a development comparison, not a retrospective switch based on the test. Every cost-table candidate uses the same sigmoid method for comparability, whereas the chosen candidate also has an isotonic comparison. Method choice and threshold optimization share the policy-selection sample, so optimized development cost is optimistic. CV scores for tuned candidates reuse training-selected hyperparameters and are descriptive, not nested-CV estimates. The final test had already been inspected before this revision; repeated reporting is regression evidence, not a new untouched holdout.

![nonlinear_transaction_amount.png](reports/nonlinear_transaction_amount.png)

![nonlinear_transactions_last_1h.png](reports/nonlinear_transactions_last_1h.png)

![nonlinear_transactions_last_24h.png](reports/nonlinear_transactions_last_24h.png)

![nonlinear_distance_from_usual_location_km.png](reports/nonlinear_distance_from_usual_location_km.png)

![nonlinear_account_age_days.png](reports/nonlinear_account_age_days.png)

![nonlinear_merchant_risk_score.png](reports/nonlinear_merchant_risk_score.png)

![nonlinear_ip_risk_score.png](reports/nonlinear_ip_risk_score.png)

![nonlinear_hour.png](reports/nonlinear_hour.png)

![dependence_transaction_amount.png](reports/dependence_transaction_amount.png)

![dependence_distance_from_usual_location_km.png](reports/dependence_distance_from_usual_location_km.png)

![dependence_merchant_risk_score.png](reports/dependence_merchant_risk_score.png)

![dependence_transactions_last_1h.png](reports/dependence_transactions_last_1h.png)

![interaction_new_device_distance_from_usual_location_km.png](reports/interaction_new_device_distance_from_usual_location_km.png)

![interaction_merchant_risk_score_transaction_amount.png](reports/interaction_merchant_risk_score_transaction_amount.png)

![interaction_authentication_level_channel.png](reports/interaction_authentication_level_channel.png)

![calibration_methods.png](reports/calibration_methods.png)

In [16]:
print(pd.read_csv('reports/calibration_before_after.csv').to_string(index=False))
print(pd.read_csv('reports/candidate_cv_summary.csv').to_string(index=False))

      method  roc_auc  average_precision    brier  log_loss  calibration_in_large  calibration_slope  expected_calibration_error
uncalibrated 0.638982           0.220790 0.105811  0.388316             -0.007767           0.256357                    0.050786
     sigmoid 0.638982           0.220790 0.104184  0.360464             -0.001676           2.566945                    0.061409
    isotonic 0.624673           0.185057 0.105291  0.627370             -0.002043           0.000665                    0.029770
                          model  ap_mean    ap_sd  roc_mean   roc_sd
         extra_trees_plain_base 0.166209 0.017261  0.575486 0.018551
   extra_trees_plain_engineered 0.208756 0.018997  0.614282 0.007380
   extra_trees_tuned_engineered 0.206611 0.010095  0.619060 0.012862
      extra_trees_weighted_base 0.165657 0.007993  0.567264 0.012860
extra_trees_weighted_engineered 0.187727 0.015984  0.581349 0.027181
              forest_plain_base 0.189927 0.038876  0.604572 0.041202
 